In [1]:
import warnings
warnings.filterwarnings('ignore')

# for interactive 
#from itables import init_notebook_mode
#init_notebook_mode(all_interactive = True)
#import itables.options as opt
#opt.maxBytes = 2**20
#opt.maxColumns= 0

import pandas as pd 
#pd.options.display.float_format = '{:.2%}'.format
from datetime import date, timedelta
import numpy as np
from os import path 

import pyfolio as pf
from pypfopt.expected_returns import returns_from_prices
from pypfopt import plotting
import ffn
import riskfolio as rp

import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
%cd /Users/safishajjouz/GitHub/QuantitativePortfolioManagement

/Users/safishajjouz/GitHub/QuantitativePortfolioManagement


In [3]:
from myPortfolioManagement.myData import * 
from myPortfolioManagement.myPortfolioSelection import *
from myPortfolioManagement.myPortfolioOptimisation import *
from myPortfolioManagement.myPerformanceAnalytics import *

In [5]:
# download from Yahoo 
# Set dates 
start_date = '2006-01-01'
end_date = date.today() -  timedelta(days=1)
end_date = end_date.strftime("%Y-%m-%d") 

df_bench = get_stock_prices(yahoo_tickers = ['^GSPC', '^IXIC'], 
                 start_date = start_date,
                 end_date = end_date, 
                 time_interval = 'daily', 
                wide_format = True) 

get_stock_prices took 671.21ms


In [7]:
df_bench = df_bench ['^GSPC']
df_bench

Date
2006-01-03    1268.800049
2006-01-04    1273.459961
2006-01-05    1273.479980
2006-01-06    1285.449951
2006-01-09    1290.150024
                 ...     
2023-01-09    3892.090088
2023-01-10    3919.250000
2023-01-11    3969.610107
2023-01-12    3983.169922
2023-01-13    3999.090088
Name: ^GSPC, Length: 4288, dtype: float64

In [12]:
dataframe = get_fidelity_prices(filter_date = '2006-01-01')
#dataframe = dataframe[dataframe['Asset_Class'].isin(['Equity', 'Alternatives', 'AbsoluteAlpha'])]
dataframe = dataframe[dataframe.index<='2021-12-01']


# from long to wide 
df = dataframe.pivot_table(index='Date', 
                        columns='fund', 
                        values='price')

# benchmark
bench = ['Vanguard FTSE Dev Wld ex-UK Eq Idx £ Acc', 
         'iShares Core S&P 500 ETF USD Acc GBP',
        'Invesco EQQQ NASDAQ-100 ETF GBP'][1]

ParserError: Error tokenizing data. C error: Expected 7 fields in line 11019, saw 8


In [ ]:
df_prices = df.merge(df_bench, on = 'Date')
#df_prices = df_prices[df_prices.index<='2020-01-01']

In [ ]:
myassets = ['Scottish Mortgage Ord', 
            'Invesco EQQQ NASDAQ-100 ETF GBP',
            'Rathbone Global Opportunities S Acc', 
            'Stewart Inv APAC Ldrs Sstby B GBP Acc',
            'BlackRock Throgmorton Trust Ord',
            'HarbourVest Global Priv Equity Ord',
            'Janus Henderson Mlt-Ast AbsRet I Acc']

In [ ]:
df_myportfolio = dataframe[dataframe.fund.isin(myassets)]
df_myportfolio.head()

In [ ]:
df_stats = performance_overview(df_prices, prices = True)
df_stats

In [ ]:
#df_prices_sm = df_myassets.rolling(window=50, min_periods=1).mean()

# daily return 
ret = returns_from_prices(df_prices)
ret

In [ ]:
df_rolling = df_prices.resample('Y').mean().pct_change().rolling(3).mean()
df_rolling

In [ ]:
df_corr = df_rolling.corr()
df_corr = df_corr[df_corr<0.4]
plt.figure(figsize=(18, 10))
heatmap = sns.heatmap(df_corr, vmin=-1, vmax=1, annot=True, cmap='BrBG')
heatmap.set_title('Correlation Heatmap', fontdict={'fontsize':18}, pad=12);

In [ ]:
rp.plot_clusters(returns=ret, codependence="tail",
                      linkage='ward', k=5, max_k=10,
                      leaf_order=True, dendrogram=True, ax=None)

In [ ]:
myassets = ['Scottish Mortgage Ord', 
            'Invesco EQQQ NASDAQ-100 ETF GBP',
            #'Rathbone Global Opportunities S Acc', 
            #'Stewart Inv APAC Ldrs Sstby B GBP Acc',
            #'BlackRock Throgmorton Trust Ord',
            'HarbourVest Global Priv Equity Ord']
            #'JPM Global Macro Opportunities C Net Acc'] 

ret_training = ret[myassets]

In [ ]:
ret

## Portfolio Optimization

### Naive Allocation 


In [ ]:
df_naive = equal_weight_portfolio(ret_training)
df_naive

### Inverse Volatility

In [ ]:
df_inverse_vol = inverse_vol_portfolio(ret_training, weight_max = 0.4)
df_inverse_vol

### Risk Parity 

In [ ]:
# Building the portfolio object
df_rp = generate_rp_portfolios(ret_training.fillna(0), weight_max = 0.4)
df_rp

In [ ]:
df_HRP_portfolios = generate_HRP_portfolios(returns_training = ret_training.dropna(), 
                            weight_max = 0.40,
                            weight_min = 0.05,
                            rf = 0.01)
#df_HRP_portfolios.index = df_HRP_portfolios.index.set_names(['fund'])
#df_HRP_portfolios = df_HRP_portfolios.reset_index()
df_HRP_portfolios

In [ ]:
df_all_weights = pd.concat([df_inverse_vol, df_naive, df_HRP_portfolios, df_rp],axis = 1)

In [ ]:
#df_all_weights = df_all_weights.set_index('fund')
df_portfolio_returns = pd.DataFrame([])
for portfolio_name in list(df_all_weights.columns):
    temp = culculate_portfolio_returns(ret_training,
                                       df_all_weights.index.to_list(),
                                       df_all_weights[portfolio_name].to_list(), 
                                       portfolio_name = portfolio_name)
    df_portfolio_returns = pd.concat([df_portfolio_returns, temp], axis=1)

df_portfolio_returns

In [ ]:
# check some basic perfomance stats for each asset 
df_stats = performance_overview(df_portfolio_returns, prices = False)
df_stats

## Evaluate

In [ ]:
ret = ret.merge(df_portfolio_returns, on = 'Date')
price_index = ffn.core.to_price_index(ret, start=100)
df_all_prices = pd.melt(price_index.reset_index(), id_vars = 'Date',value_name = 'price', var_name = 'fund')

In [ ]:
rolling_window     = 3
rolling_frequency  = 'Y'
my_assets_col_name = 'fund' 
my_date_col_name   = 'Date'
price_col_name = 'price'
benchmark_name = bench

df_reward_metrics = ranking_metrics(df = df_all_prices, 
                                   rolling_window = rolling_window, 
                                   rolling_frequency = rolling_frequency,
                                   my_assets_col_name = my_assets_col_name, 
                                   my_date_col_name = my_date_col_name,
                                   price_col_name = price_col_name,
                                   benchmark_name = benchmark_name)

df_reward_metrics

In [ ]:
best_portf = 'por_HRP_MV_pearson_ledoit_ward'

In [ ]:
df_all_weights[[best_portf]]

In [ ]:
benchmark = ['Vanguard FTSE Dev Wld ex-UK Eq Idx £ Acc'] 
ret_myassets = returns_from_prices(df_prices.dropna())
ret_myassets[myassets + benchmark].columns

In [ ]:
myWeights = df_all_weights[[best_portf]].iloc[:,0].to_list()
df_portf_returns = culculate_portfolio_returns(ret_myassets[myassets], 
                                myassets_list = ret_myassets[myassets].columns.to_list(),
                                myweights_list =myWeights,
                                portfolio_name = 'myPortfolio')
df_backtest = df_portf_returns.merge(ret.drop(myassets, axis = 1), on = 'Date')
df_backtest = df_portf_returns.merge(ret, on = 'Date')

In [ ]:
col_to_keep = list(set([best_portf, bench] + myassets))

In [ ]:
price_index = ffn.core.to_price_index(ret[col_to_keep], start=100)
df_rolling_pseudo = price_index.resample('Y').mean().pct_change().rolling(3).mean().dropna()

In [ ]:
df_rolling_pseudo

In [ ]:
import plotly.express as px

In [ ]:
fig = px.scatter(df_scatter.reset_index(), x="Aver_Rolling_Std", y="Aver_Rolling_Returns", 
                 size ='Aver_Rolling_Returns', template='plotly_white', text='index',
                 color_continuous_scale=px.colors.sequential.Viridis,
                 width=900, height=800,
                 title = 'Risk Reward Backtesting: last 3 Years')
fig.add_hline(y=0.120563)
fig.add_vrect(x0=0.068116, x1=0.068116)
fig.show()

In [ ]:
df = pf.timeseries.cum_returns(ret[col_to_keep], starting_value=1)

In [ ]:
#df = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/finance-charts-apple.csv')

fig = px.line(df.reset_index(), x='Date', y=df.columns, width=1000, height=800,
              title='Time Series with Range Slider and Selectors')

fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=6, label="6m", step="month", stepmode="backward"),
            dict(count=1, label="YTD", step="year", stepmode="todate"),
            dict(count=1, label="1y", step="year", stepmode="backward"),
            dict(step="all")
        ])
    )
)
fig.show()

In [ ]:
import quantstats as qs
qs.reports.full(ret[best_portf], '^IXIC')